<a href="https://colab.research.google.com/github/AndrVel/simulative_python/blob/master/%D0%A2%D1%80%D0%B5%D1%82%D0%B8%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 3 кейс

**В этом кейсе вы будете рассчитывать:**
* retention
* rolling retention
* lifetime
* churn rate
* mau
* wau
* dau

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [254]:
from datetime import datetime, timedelta

In [255]:
!wget https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv

!wget https://gist.github.com/Vs8th/aacb80595d1d6aaa2e31eb735f8bc644/raw/entries.csv

!wget https://gist.github.com/Vs8th/0e827e9a608117345dd6585ab81e8c86/raw/metrics.txt

--2026-06-15 15:19:02--  https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.github.com (gist.github.com)... 140.82.112.3
Connecting to gist.github.com (gist.github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv [following]
--2026-06-15 15:19:03--  https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14918 (15K) [text/plain]
Saving to: ‘registrations.csv.2’

registrations.csv.2 100%[===================>]  14.57K  --.-KB/s    in 0s      

2026-06-15 15:19:03 (28.8

Файлами для работы являются `registrations.csv` и `entries.csv`. В них хранятся данные о регистрациях пользователей и входа на платформу соответственно.

### **Посчитайте Retention 15 дня (в процентах) для пользователей, зарегистрированных в январе**

Cохраните результат в переменную `retention_15_day`

**Примечание:** результат округлите до 5 знаков после запятой

In [256]:
# Ваше решение
with open("registrations.csv", "r") as registrations:

  regs = registrations.read()
registrations = {}

regs_list = [x.split(';') for x in regs.split('\n')[1:]]
for item in regs_list[:]:
  reg_date = datetime.strptime(item[1], '%Y-%m-%d')
  registrations[int(item[0])] = {'reg_date': reg_date, 'reg_mnth': reg_date.month}

# формирую словарь пользователей, зарегистрированных в январе
registrations_jan = {}
january = 1
for item in registrations:
  if registrations[item]['reg_mnth'] == january:
    registrations_jan[item] = registrations[item]


In [257]:
with open("entries.csv", "r") as entries:
  ent = entries.read()

# Формирую список со всеми входами.
ent_list = [x.split(';') for x in ent.split('\n')[1:]]

# иду по списку заходов, проверяю по user_id наличие в словаре registrations_jan и что это 15-й день.
entries_15 = [] #список, куда добавляется user_id тех, кто заходил в 15 день.
for item in ent_list:
  user_id = int(item[0])
  entry_date = datetime.strptime(item[1], '%Y-%m-%d')
  if (user_id in registrations_jan and
      entry_date - registrations_jan[user_id]['reg_date'] == timedelta(days=15)):
    entries_15.append(user_id)

# преобразую список в сет, чтобы избавиться от дублей
retention_15_day = round(len(set(entries_15)) / len(registrations_jan) * 100, 5)
retention_15_day

54.65116

In [258]:
# from google.colab import drive
# drive.mount('/content/drive')

In [259]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
# Открываем файл с правильными ответами
with open('metrics.txt', 'r') as f:
    answers = f.read().split('\n')

correct_answer = float(answers[0])

try:
    assert retention_15_day == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Rolling-retention 30 дня (в процентах) для пользователей из той же когорты**

Сохраните результат в переменную `rolling_retention`

**Примечание:** результат округлите до 5 знаков после запятой

In [260]:
# Ваше решение
# Список с user_id, которые заходили в 30-день или в любой день после для зарегистрированных в январе
entries_30 = []

# прохожу по списку заходов
for item in ent_list:
  user_id = int(item[0])
  entry_date = datetime.strptime(item[1], '%Y-%m-%d')
  if (user_id in registrations_jan and
      entry_date - registrations_jan[user_id]['reg_date'] >= timedelta(days=30)):
    entries_30.append(user_id)
rolling_retention = round(len(set(entries_30)) / len(registrations_jan) * 100, 5)
rolling_retention



29.06977

In [261]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[1])

try:
    assert rolling_retention == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Lifetime по всем пользователям, посчитанный как интеграл от n-day retention**

Сохраните результат в переменную `lifetime`

**Примечание:** результат округлите до 5 знаков после запятой

In [262]:
# расчет n-day retention
def n_days_retention(registrations, entries, n_days):
  # Общее количество зарегистрированных - qty_registered
  qty_registered = len(set(registrations.keys()))
  #
  days_entries = {}
  for item in entries:
    days_after_reg = (item[1] - registrations[item[0]]['reg_date'])
    if days_after_reg == timedelta(days=n_days):
      days_entries[item[0]] = days_after_reg
  qty_entered = len(days_entries)

  return qty_entered/qty_registered

In [263]:
with open('entries.csv', 'r') as entries:
  ent = entries.read()
entrs = ent.split("\n")[1:]
# формирую список entries с учетом парсинга даты и приведения user_id к целому.
entries = [[int(x.split(";")[0]), datetime.strptime(x.split(";")[1], '%Y-%m-%d')] for x in entrs]


In [264]:
# расчет lifetime
n=0
lifetime = n_days_retention(registrations, entries, 0)
n = 1
while n_days_retention(registrations, entries, n) > 0:
  lifetime += n_days_retention(registrations, entries, n)
  n += 1
lifetime = round(lifetime, 5)

In [265]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[2])

try:
    assert lifetime == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Churn rate 29 дня (в долях), посчитанный по всем пользователям**

Сохраните результат в переменную `churn_29`.  
Если будете считать CR от Retention, ведите расчет от Rolling Retention, таким образом, мы получим всех, кто не заходил в 29 день и после. Ведя расчет от обычного Retention, наоборот - получим только CR в 29 день.


In [266]:
# Ваше решение
# n-days rolling retantion - доля тех, кто заходил в n-й день или после от количества зарегистрированных пользователей.
def n_churn(registrations, entries, n_days):
  reged_qty = len(registrations)

  # убираю дубликаты из entries чтобы не считать несколько входов одним пользователем в день.
  entries_unique = list(set([(x[0], x[1]) for x in entries]))

  retained = {}
  for item in entries_unique:
    days_diff = (item[1] - registrations[item[0]]['reg_date']).days
    if days_diff >= n_days:
      retained[item[0]] = 1

  churn_29 = 1 - len(retained)/reged_qty

  return churn_29



In [267]:
churn_29 = n_churn(registrations, entries, 29)
churn_29

0.509

In [268]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[3])

try:
    assert churn_29 == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau за последний месяц/неделю/день записей**

Сохраните результат в переменные `dec_mau`, `dec_wau`, `dec_dau` соответственно

**Примечание:** последний месяц записей - декабрь. Поэтому `mau` рассчитываем для декабря (2021 года), для `wau` берем последнюю неделю - с 25 по 31 декабря, и для `dau` соответственно последний день - 31 декабря.

In [269]:
# Ваше решение
# Уникальные значения входа
entries_unique = list(set([(x[0], x[1]) for x in entries]))
# сет user_id, входивших в декабре
mau_set = set()
for item in entries_unique:
  if item[1] >= datetime(2021, 12, 1):
    mau_set.add(item[0])
dec_mau = len(mau_set)
dec_mau

133

In [270]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[4])

try:
    assert dec_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [271]:
# Ваше решение
dec_week = set()
for item in entries_unique:
  if item[1] >= datetime(2021, 12, 25) and item[1] <= datetime(2021, 12, 31):
    dec_week.add(item[0])
dec_wau = len(dec_week)
dec_wau

84

In [272]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[5])

try:
    assert dec_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [273]:
# Ваше решение
dec_day = set()
for item in entries_unique:
  if item[1] == datetime(2021, 12, 31):
    dec_day.add(item[0])
dec_dau = len(dec_day)
dec_dau

47

In [274]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[6])

try:
    assert dec_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau усредненные**

Сохраните результат в переменные `avg_mau`, `avg_wau`, `avg_dau` соответственно

**Примечание:** результаты округлите до 5 знаков после запятой

In [275]:
# Ваше решение
entries_unique
# создаю список кортежей
monthly_entry = []
for item in entries_unique:
  monthly_entry.append((item[0], item[1].month, item[1].year))

# Удаляю дубликаты входов. каждый user_id должен учитываться только один раз.
monthly_entry = list(set(monthly_entry))
monthly_dict = {}
for item in monthly_entry:
  if item[1] not in monthly_dict:
    monthly_dict[item[1]] = 1
  else:
    monthly_dict[item[1]] += 1
avg_mau = round(sum(monthly_dict.values()) / len(monthly_dict), 5)

avg_mau

102.58333

In [276]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[7])

try:
    assert avg_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [277]:
# Ваше решение
# создаю список кортежей
weekly_entry = []
for item in entries_unique:
  weekly_entry.append((item[0], item[1].isocalendar().week))
# Удаляю дубликаты входов. каждый user_id должен учитываться только один раз.
weekly_entry = list(set(weekly_entry))
weekly_count = {}
for item in weekly_entry:
  if item[1] in weekly_count:
    weekly_count[item[1]] += 1
  else:
    weekly_count[item[1]] = 1

avg_wau = round(sum(weekly_count.values()) / len(weekly_count), 5)
avg_wau

89.86792

In [278]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[8])

try:
    assert avg_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [279]:
datetime(2021, 8, 3, 0, 0)

datetime.datetime(2021, 8, 3, 0, 0)

In [280]:
# Ваше решение
entries_unique
daily = {}
for item in entries_unique:
  if item[1] in daily:
    daily[item[1]] += 1
  else:
    daily[item[1]] = 1

avg_dau = round(sum(daily.values()) / len(daily), 5)
avg_dau

40.5589

In [281]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[9])

try:
    assert avg_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
